In [2]:
"""
NLP Resume Classifier + Resume-to-Job-Description Matcher
=============================================================
Dataset: Kaggle "Resume Dataset" by snehaanbhawal
Download: https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset
Columns: ID, Resume_str, Resume_html, Category (auto-detected below —
works with either this dataset or ones using plain "Resume"/"Category"
column names).
Place as: data/Resume.csv

Part A: train a classifier to predict a resume's job category from its
text (a real supervised NLP benchmark with metrics).
Part B (in app.py): use sentence embeddings to score how well a resume
matches a specific job description, and flag missing skills — the
practical tool.
"""

import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib

RANDOM_STATE = 42

# -----------------------------------------------------------------
# 1. LOAD DATA
# -----------------------------------------------------------------
# Handles either the "UpdatedResumeDataSet.csv" naming (Category, Resume)
# or the "snehaanbhawal/resume-dataset" naming (Category, Resume_str) —
# auto-detects which columns are present so you don't need to rename
# anything manually regardless of which Kaggle dataset you downloaded.
df = pd.read_csv("data/Resume.csv")
print("Columns found:", list(df.columns))

resume_col = next((c for c in ["Resume", "Resume_str", "resume_str", "Text", "text"] if c in df.columns), None)
category_col = next((c for c in ["Category", "category"] if c in df.columns), None)

if resume_col is None or category_col is None:
    raise ValueError(
        f"Couldn't auto-detect the resume text / category columns. "
        f"Found columns: {list(df.columns)}. Edit resume_col/category_col above manually."
    )

df = df.rename(columns={resume_col: "Resume", category_col: "Category"})
print(df.shape)
print(df["Category"].value_counts())


Columns found: ['ID', 'Resume_str', 'Resume_html', 'Category']
(2484, 4)
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64


In [4]:
# -----------------------------------------------------------------
# 2. EDA
# -----------------------------------------------------------------
plt.figure(figsize=(10, 6))
df["Category"].value_counts().plot(kind="barh")
plt.title("Resume count by job category")
plt.tight_layout()
plt.savefig("outputs/category_distribution.png", bbox_inches="tight")
plt.close()

df["word_count"] = df["Resume"].apply(lambda x: len(str(x).split()))
print("\nResume length stats:\n", df["word_count"].describe())



Resume length stats:
 count    2484.000000
mean      811.325684
std       371.006906
min         0.000000
25%       651.000000
50%       757.000000
75%       933.000000
max      5190.000000
Name: word_count, dtype: float64


In [5]:
# -----------------------------------------------------------------
# 3. TEXT CLEANING
# -----------------------------------------------------------------
# WHY each step: resumes are copy-pasted from PDFs/Word docs and are
# full of noise that adds no signal for classification — URLs, emails,
# phone numbers, and special characters just add sparse, useless
# vocabulary to the TF-IDF matrix.
def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", " ", text)          # URLs
    text = re.sub(r"\S+@\S+", " ", text)                  # emails
    text = re.sub(r"[^a-zA-Z\s]", " ", text)              # punctuation/numbers
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

df["clean_resume"] = df["Resume"].apply(clean_text)


In [6]:
# -----------------------------------------------------------------
# 4. TRAIN / TEST SPLIT
# -----------------------------------------------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_resume"], df["Category"],
    test_size=0.2, stratify=df["Category"], random_state=RANDOM_STATE
)


In [7]:
# -----------------------------------------------------------------
# 5. TF-IDF VECTORIZATION
# -----------------------------------------------------------------
# WHY max_features + ngrams: resumes have huge vocabularies (names,
# tools, company names); capping features keeps the model from
# overfitting to rare one-off tokens, and bigrams catch phrases like
# "machine learning" that a single word loses.
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)


In [8]:
# -----------------------------------------------------------------
# 6. MODEL COMPARISON
# -----------------------------------------------------------------
models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Linear SVM": LinearSVC(random_state=RANDOM_STATE),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="weighted")
    results[name] = {"accuracy": acc, "f1_weighted": f1}
    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc:.4f}, Weighted F1: {f1:.4f}")

best_name = max(results, key=lambda k: results[k]["f1_weighted"])
best_model = models[best_name]
print(f"\nBest model: {best_name}")
print(classification_report(y_test, best_model.predict(X_test)))


--- Multinomial Naive Bayes ---
Accuracy: 0.5654, Weighted F1: 0.5308

--- Logistic Regression ---
Accuracy: 0.6680, Weighted F1: 0.6520

--- Linear SVM ---
Accuracy: 0.7404, Weighted F1: 0.7330

Best model: Linear SVM
                        precision    recall  f1-score   support

            ACCOUNTANT       0.72      0.88      0.79        24
              ADVOCATE       0.67      0.75      0.71        24
           AGRICULTURE       0.89      0.62      0.73        13
               APPAREL       0.54      0.37      0.44        19
                  ARTS       0.56      0.43      0.49        21
            AUTOMOBILE       0.75      0.43      0.55         7
              AVIATION       0.86      0.75      0.80        24
               BANKING       0.72      0.78      0.75        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.71      0.83      0.77        24
                  CHEF       0.86      0.79      0.83        24
          C

In [9]:
# Confusion matrix for the best model
cm = confusion_matrix(y_test, best_model.predict(X_test), labels=best_model.classes_)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, xticklabels=best_model.classes_, yticklabels=best_model.classes_, cmap="Blues")
plt.title(f"Confusion Matrix — {best_name}")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("outputs/confusion_matrix.png", bbox_inches="tight")
plt.close()


In [10]:
# -----------------------------------------------------------------
# 7. SAVE ARTIFACTS
# -----------------------------------------------------------------
joblib.dump(best_model, "outputs/resume_classifier.pkl")
joblib.dump(vectorizer, "outputs/tfidf_vectorizer.pkl")

print("\nDone. Model + vectorizer saved to outputs/")
print("\nNext: run build_skills_list.py (optional) then `streamlit run app.py`")



Done. Model + vectorizer saved to outputs/

Next: run build_skills_list.py (optional) then `streamlit run app.py`
